# Visium HD - SCA020 100M

## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import gc
import spatialdata_plot
from spatialdata.transformations import get_transformation, remove_transformation, set_transformation, Translation, Sequence, Identity

matplotlib.use("Agg")

import logging
logging.getLogger('ome_zarr').setLevel(logging.ERROR) # This mutes unnecessary Zarr warning (see: https://github.com/scverse/spatialdata/issues/400)

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _visium_hd import visium_hd

## Create sdata

In [4]:
dataset_id = "SCA020"

path_outs = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/subsampled_100M/SpaceRanger/outs'
path_img = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/4/IMAGES/{dataset_id}_HE_hires.tif'
output_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/subsampled_100M/harpy/sdata.zarr'

pipeline_name = 'SpaceRanger'
species = 'human'
tissue_annotation_file = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/{dataset_id}/tissue_annotation/tissue.geojson'

In [5]:
# sdata = sd.read_zarr(output_path)
# sdata

In [6]:
sdata = visium_hd(
    path=path_outs, 
    dataset_id=dataset_id,
    filtered_counts_file=False,
    bin_size=[2, 8, 16, 32], 
    bins_as_squares=True, 
    fullres_image_file=path_img,
    load_all_images=True,
    output=output_path
)
sdata

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data/groups/technologies/spati

INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column named 'relabeled_location_id'.                 
INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column named 'relabeled_location_id'.                 
INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column name

<tifffile.TiffFile 'SCA020_HE_hires.tif'> ImageJ series metadata invalid or corrupted file


SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/sdata.zarr
├── Images
│     ├── 'SCA020_cytassist_image': DataArray[cyx] (3, 3000, 3201)
│     ├── 'SCA020_full_image': DataTree[cyx] (3, 23871, 28962), (3, 11935, 14481), (3, 5967, 7240), (3, 2983, 3620), (3, 1491, 1810)
│     ├── 'SCA020_hires_image': DataArray[cyx] (3, 4946, 6000)
│     └── 'SCA020_lowres_image': DataArray[cyx] (3, 495, 600)
├── Labels
│     ├── 'SCA020_square_002um_labels': DataArray[yx] (3350, 3350)
│     ├── 'SCA020_square_008um_labels': DataArray[yx] (838, 838)
│     ├── 'SCA020_square_016um_labels': DataArray[yx] (419, 419)
│     └── 'SCA020_square_032um_labels': DataArray[yx] (210, 210)
├── Shapes
│     ├── 'SCA020_square_002um': GeoDataFrame shape: (10337956, 1) (2D shapes)
│     ├── 'SCA020_square_008um': GeoDataFrame shape: (694856, 1) (2D shapes)
│     ├── 'SCA020_square_016um': Ge

In [7]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

SpaceRanger levels: ['square_002um', 'square_008um', 'square_016um', 'square_032um']


In [8]:
# Assert whether number of shapes matches number of label IDs
import dask.array as da

for level in levels:
    unique_ids = da.unique(sdata.labels[f"{dataset_id}_{level}_labels"].data).compute()
    unique_ids = unique_ids[unique_ids != 0]

    number_of_label_IDs = len(unique_ids)
    number_of_shapes = len(sdata.shapes[f"{dataset_id}_{level}"])

    assert number_of_shapes == number_of_label_IDs, (f"{level}: Number of shapes ({number_of_shapes}) does not match number of label IDs ({number_of_label_IDs})")

# NOTE: Doing -1 to account for background label ID 0 does not work if there is no background.

## Calculate pixel size

In [9]:
# Determine pixel size in microns
import tifffile

with tifffile.TiffFile(path_img) as tif:
    xres = tif.pages[0].tags["XResolution"].value  # (numerator, denominator)
    yres = tif.pages[0].tags["YResolution"].value
    unit = tif.pages[0].tags["ResolutionUnit"].value

    px_per_cm_x = xres[0] / xres[1]
    px_per_cm_y = yres[0] / yres[1]

    if unit.name == "CENTIMETER":
        microns_per_pixel_x = 10_000 / px_per_cm_x
        microns_per_pixel_y = 10_000 / px_per_cm_y
    elif unit.name == "INCH":
        microns_per_pixel_x = 25_400 / px_per_cm_x
        microns_per_pixel_y = 25_400 / px_per_cm_y
    else:
        raise ValueError(f"Unknown ResolutionUnit: {unit}")

if microns_per_pixel_x == microns_per_pixel_y:
    microns_per_pixel = microns_per_pixel_x
    print(f"Pixel size: {microns_per_pixel:.3f} µm")
else:
    print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")


Pixel size: 0.442 µm


## Hexagonal tesselation

In [10]:
custom_bins = ['hexagonal_grid_20um', 'hexagonal_grid_120um'] # Names refer to the diagonal of the hexagon

In [11]:
min_x, min_y, max_x, max_y = sdata.shapes[f'{dataset_id}_square_002um']['geometry'].total_bounds
shape = (max_y - min_y, max_x - min_x)

transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True)
new_transformations = {}
        
for coordinate_system, transformation in transformations.items():
    translation = Translation([min_x, min_y], axes=("x", "y"))
    if isinstance(transformation, Identity):
        new_transformations[coordinate_system] = translation
    else:
        new_transformations[coordinate_system] = Sequence([translation, transformation])

for custom_bin in custom_bins:
    gc.collect()
    
    size = (int(custom_bin.split("hexagonal_grid_")[1].split("um")[0])/2) / microns_per_pixel # Radius of the hexagon in pixels
    
    # Create hexagonal grids
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape,
        size=size,
        output_shapes_layer=f"{custom_bin}_shapes", 
        output_labels_layer=f"{custom_bin}_labels", 
        grid_type="hexagon", 
        offset=(0, 0), 
        chunks=2048, 
        client=None, 
        transformations=new_transformations,
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

    # Rebin data based on hexagonal grid
    sdata = hp.tb.bin_counts( 
        sdata,
        table_layer="square_002um",
        labels_layer=f"{custom_bin}_labels",
        output_layer=f"{custom_bin}_table",
        append = False,
        to_coordinate_system = dataset_id,
        chunks = 50000,
        overwrite=True,
    )

2026-01-29 11:44:43.073 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'hexagonal_grid_20um_labels'
2026-01-29 11:45:00.743 | INFO     | harpy.image._image:_get_translation:55 - There seems to be more than one coordinate system defined on the provided spatial element ('['SCA020', 'SCA020_downscaled_hires', 'SCA020_downscaled_lowres']'). We only consider the coordinate sytem specified via parameter 'to_coordinate_system': 'SCA020'.
2026-01-29 11:45:00.757 | INFO     | harpy.table._allocation:_aggregate:488 - Calculating cell counts.
2026-01-29 11:45:31.842 | INFO     | harpy.table._allocation:bin_counts:363 - 
 Number of spots per bin  Frequency
                       1        243
                       2        493
                       3        697
                       4        906
                       5       1058
                       6       1099
                       7       1057
                       8       1014
                       9        

## Specify plotting parameters

In [12]:
DPI = 300
MAX_DIM_FOR_PLOTS = 10000

HE_image = f'{dataset_id}_full_image'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale2
figsize: (24.133333333333333, 19.89)
plots_path: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots


## Create crop for zoomed-in plots

In [13]:
min_x = sdata.images[HE_image]['scale0']['image'].shape[2] / 2 # Center of image
max_x = min_x + round(200 / microns_per_pixel) # 200 micrometer
min_y = sdata.images[HE_image]['scale0']['image'].shape[1] / 2 # Center of image
max_y = min_y + round(200 / microns_per_pixel) # 200 micrometer

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y], 
    max_coordinate=[max_x, max_y], 
    axes=("x", "y"), 
    target_coordinate_system=dataset_id
)

figsize_crop = (3,3)

## Plot H&E

In [14]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/HE.png


## Plot Space Ranger spots

In [15]:
for level in levels:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - {level}"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
            element=f'{dataset_id}_{level}', fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"{pipeline_name} - {level}"
    
    sdata_crop.pl.render_images(
        element=HE_image, scale="full"
    ).pl.render_shapes(
            element=f'{dataset_id}_{level}', fill_alpha=0, outline_width=0.25, outline_alpha=1, outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize_crop,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

# NOTE: Spots are represented true to scale

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/SpaceRanger_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/crop_SpaceRanger_spots.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'N

## Plot hexagonal spots

In [16]:
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata_crop.pl.render_images(
        element=HE_image, scale="full"
    ).pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize_crop,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Spots are represented true to scale


INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/hexagonal_grid_20um.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/crop_hexagonal_grid_20um.png
INFO     Value for parame

## Plot Space Ranger tissue annotation

In [17]:
for level in levels:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} {level} - {pipeline_name} tissue annotation"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue', table_name=level, scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_tissue_annotation.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/SpaceRanger_tissue_annotation.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_008um/SpaceRanger_tissue_annotation.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_016um/SpaceRanger_tissue_annotation.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_032um/SpaceRanger_tissue_annotation.png


## Create bounding box for total capture area

In [18]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_square_002um'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True),
    overwrite = True,
)

In [19]:
# Export bbox GeoJSON
sdata.shapes[f'{dataset_id}_capture_area'].to_file(os.path.join(os.path.dirname(output_path), f'{dataset_id}_capture_area.geojson'), driver="GeoJSON")

In [20]:
# Plot capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.05)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/capture_area_bbox.png


In [21]:
# Plot capture area bbox
for level in ['square_002um']:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} {level} - Capture area bounding box"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_shapes(
        element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
    ).pl.render_shapes(
        element=f'{dataset_id}_{level}', fill_alpha=0, outline_width=1, outline_alpha=1, outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'capture_area_bbox.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/capture_area_bbox.png


In [22]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

Area of capture area: 44.89 mm²


In [23]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

Dimensions of capture area bbox: 0.67 cm x 0.67 cm


## Add tissue annotation

In [24]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_square_002um'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2026-01-29 12:14:31.345 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'SCA020_tissue_mask'


In [25]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'Qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/Qupath_tissue_annotation.png


In [26]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = level, 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = f"{custom_bin}_table", 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

2026-01-29 12:17:17.120 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_002um' already exists. Overwriting...
2026-01-29 12:18:39.503 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_008um' already exists. Overwriting...
2026-01-29 12:19:33.928 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_016um' already exists. Overwriting...
2026-01-29 12:20:28.891 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_032um' already exists. Overwriting...
2026-01-29 12:21:31.634 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table' already exists. Overwriting...
2026-01-29 12:22:21.290 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table' already exists. Overwriting...


In [27]:
# Check spots in tissue
for level in levels:
    gc.collect()

    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{level} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=level, scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/Qupath_tissue_annotation_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_008um/Qupath_tissue_annotation_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_016um/Qupath_tissue_annotation_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_032um/Qupath_tissue_annotation_spots.png


In [28]:
# Check spots in tissue
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{custom_bin} - Qupath tissue annotation"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{custom_bin}_labels", fill_alpha=0.4, contour_px=5, outline_alpha=0, color='in_tissue_annotation', table_name=f"{custom_bin}_table", scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'Qupath_tissue_annotation_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Qupath_tissue_annotation_spots.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_120um/Qupath_tissue_annotation_spots.png


## QC

In [29]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = level,
        species = species
    )

for custom_bin in custom_bins:
    gc.collect()
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{custom_bin}_table",
        species = species
    )

2026-01-29 12:28:20.541 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_002um' already exists. Overwriting...
2026-01-29 12:29:45.894 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_008um' already exists. Overwriting...
2026-01-29 12:30:48.466 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_016um' already exists. Overwriting...
2026-01-29 12:31:32.474 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'square_032um' already exists. Overwriting...
2026-01-29 12:32:11.951 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table' already exists. Overwriting...
2026-01-29 12:33:09.075 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table' already exists. Overwriting...


In [30]:
# Plot QC metrics
for level in levels:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[level].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=level, cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/Scanpy_QC_n_genes_by_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/Scanpy_QC_total_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/Scanpy_QC_pct_counts_in_top_50_genes.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/Scanpy_QC_total_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/square_002um/Scanpy_QC_pct_counts_mt.png
Plot saved at: /data/g

In [31]:
# Plot QC metrics
for custom_bin in custom_bins:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, custom_bin)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{custom_bin}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{custom_bin}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, scale=best_scale, table_name=f"{custom_bin}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_n_genes_by_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts_in_top_50_genes.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_c

## Leiden clustering

In [32]:
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY

for custom_bin in custom_bins:
    gc.collect()
    
    print(f'running: {custom_bin}', flush=True)

    labels_layer=f"{custom_bin}_labels"
    shapes_layer=f"{custom_bin}_shapes"
    table_layer=f"{custom_bin}_table"
    
    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)
    
    # Only keep spots in tissue for clustering
    adata = sdata.tables[table_layer][sdata.tables[table_layer].obs['in_tissue_annotation'] == True]
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=f"{table_layer}_processed",
        region=adata.obs[_REGION_KEY].cat.categories.to_list(),
        overwrite=True,
    )
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False,
        labels_layer=labels_layer,
        table_layer=f"{table_layer}_processed",
        output_layer=f"{table_layer}_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps=50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.75, 1, 1.25]
    calculate_umap = True
    calculate_neighbors = True

    for res in leiden_resolutions:
        gc.collect()

        res_str = str(res).replace('.', '')
        print(f'Leiden resolution: {res}', flush=True)
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_processed",
            output_layer=f"{table_layer}_processed",
            calculate_umap=calculate_umap,
            calculate_neighbors=calculate_neighbors,
            rank_genes=False,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        calculate_umap = False # Only calculate UMAP once
        calculate_neighbors = False # Only calculate neighbors once

        # Plot Leiden clustering
        gc.collect()
        
        with matplotlib.rc_context({'legend.markerscale': 2}):
            fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

            title = f"Leiden {res}"

            sdata.pl.render_images(
                element=HE_image, scale=best_scale
            ).pl.render_labels(
                element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, cmap="rainbow", color=f'leiden_{res_str}', table_name=f"{table_layer}_processed", scale=best_scale
            ).pl.show(
                coordinate_systems=dataset_id,
                title=title,
                figsize=figsize,
                colorbar=False,
                legend_fontsize=rel_fontsize(fig, 0.015), 
                legend_fontweight='bold', 
                legend_loc='right margin',
                na_in_legend=False,
                dpi=DPI,
                ax=ax,
            )

            ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
            ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
            ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
            ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

            save_path = os.path.join(level_plots_path, f'Leiden_{res}.png')
            fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

            plt.close(fig)

            print("Plot saved at:", save_path, flush=True)
        
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(level_plots_path, f'Leiden_{res}')
        )

running: hexagonal_grid_20um


2026-01-29 12:55:15.586 | INFO     | harpy.table._preprocess:preprocess:314 - Calculating cell size from provided labels_layer 'hexagonal_grid_20um_labels'
2026-01-29 12:55:18.483 | INFO     | harpy.table._preprocess:preprocess:334 - Saving non preprocessed data matrix to the AnnData '.layers[raw_counts]'.
2026-01-29 12:56:18.019 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Leiden resolution: 0.75


2026-01-29 13:12:53.813 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_20um/Leiden_0.75.png
Leiden resolution: 1


...

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_120um/Leiden_0.75.png
Leiden resolution: 1


2026-01-29 13:58:43.064 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_120um/Leiden_1.png
Leiden resolution: 1.25


2026-01-29 14:01:40.649 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/plots/hexagonal_grid_120um/Leiden_1.25.png


## Spots metrics

In [33]:
gc.collect()

20113

In [34]:
tissue_positions = pd.read_parquet(os.path.join(path_outs, 'binned_outputs/square_002um/spatial/tissue_positions.parquet'))
tissue_positions

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_002um_00000_00000-1,0,0,0,21355.944925,7782.924470
1,s_002um_00000_00001-1,0,0,1,21355.894767,7787.446402
2,s_002um_00000_00002-1,0,0,2,21355.844610,7791.968335
3,s_002um_00000_00003-1,0,0,3,21355.794453,7796.490267
4,s_002um_00000_00004-1,0,0,4,21355.744296,7801.012200
...,...,...,...,...,...,...
11222495,s_002um_03349_03345-1,0,3349,3345,6042.597982,22742.060287
11222496,s_002um_03349_03346-1,0,3349,3346,6042.547915,22746.581926
11222497,s_002um_03349_03347-1,0,3349,3347,6042.497848,22751.103564
11222498,s_002um_03349_03348-1,0,3349,3348,6042.447782,22755.625202


In [35]:
# Total number of spots in capture area
print(f"Total number of spots in capture area: {len(tissue_positions):,}")

Total number of spots in capture area: 11,222,500


In [36]:
# Total number of spots in tissue
tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
mask_shape = tissue_mask.shape

tissue_positions["in_tissue_annotation"] = tissue_positions.apply(
    lambda row: assign_region((row["pxl_col_in_fullres"], row["pxl_row_in_fullres"]), tissue_mask, mask_shape), axis=1
)

total_n_spots_in_tissue = len(tissue_positions[tissue_positions['in_tissue_annotation'] == 1])

print(f"Total number of spots in tissue: {total_n_spots_in_tissue:,}")

Total number of spots in tissue: 10,013,378


In [37]:
# Total number of spots in table
n_spots = len(sdata.tables['square_002um'].obs)
print(f"Number of spots in table: {n_spots:,}")

Number of spots in table: 10,337,956


In [38]:
# Number of spots in tissue (in table)
n_spots_in_tissue = len(sdata.tables['square_002um'][sdata.tables['square_002um'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue:,}")

Number of spots in tissue: 10,001,890


In [39]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in tissue_positions['barcode']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in tissue_positions[tissue_positions['in_tissue_annotation'] == 1]['barcode']:
        f.write(f"{item}\n")      

In [40]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 40.05 mm²


## Capture effeciency metrics

In [41]:
gc.collect()

108

In [42]:
def row_sums(X):
    """Return 1D array of row sums without densifying."""
    return np.asarray(X.sum(axis=1)).ravel()

def col_sums(X):
    """Return 1D array of column sums without densifying."""
    return np.asarray(X.sum(axis=0)).ravel()

In [43]:
probe_set_df = pd.read_csv(f"{path_outs}/probe_set.csv", comment='#')
probe_set_df

,gene_id,probe_seq,probe_id,included,region
0,ENSG00000000003,GGTGACACCACAACAATGCAACGTATTTTGGATCTTGTCTACTGCA...,ENSG00000000003|TSPAN6|8eab823,True,spliced
1,ENSG00000000003,TCTGCATCTCTCTGTGGAGTACAATCTTCAAGTTTACAGCAACTCT...,ENSG00000000003|TSPAN6|9d7fe51,True,unspliced
2,ENSG00000000003,AAAGCTGTTCTTAATCTCATGTCTGAAAACAAATCCTACGATGGCA...,ENSG00000000003|TSPAN6|d2b5833,True,spliced
3,ENSG00000000005,CGTGACGGGTCTTCTCTACTTTCACTTGAGGGACCACCCACTGTTC...,ENSG00000000005|TNMD|7790621,True,unspliced
4,ENSG00000000005,GCCTCGACGGCAGTAAATACAACAATAACCTCTCTCATCCAGCATG...,ENSG00000000005|TNMD|923f04b,True,unspliced
...,...,...,...,...,...
54575,DEPRECATED_ENSG00000278803,CCTGGAGTAGGAGCAATGGCCTTGTGTTTCCGTGACCTGTCAGACC...,DEPRECATED_ENSG00000278803|AC236972.4|53ec843,False,unspliced
54576,DEPRECATED_ENSG00000278803,ACCCTCTCCGGGGTCCTGAATGTTTCCCGAGAAGGGAGAACTTTCC...,DEPRECATED_ENSG00000278803|AC236972.4|b4ceb8b,False,unspliced
54577,DEPRECATED_ENSG00000283967,CAGCAGGGTTGTCATCCTCTGAGCTTCCTCCGCATCCACGGTGGGC...,DEPRECATED_ENSG00000283967|TAF11L8|5800ed2,False,unspliced
54578,DEPRECATED_ENSG00000284873,AGCGCAGGTCCATATCAAGGAATGCAGATAAAAGAGCCCTTTGAAC...,DEPRECATED_ENSG00000284873|OOSP1|b83341e,False,unspliced


In [44]:
# Probe set metrics
print(f"Number of gene IDs in table: {probe_set_df['gene_id'].nunique():,}")
print(f"Number of included gene IDs in table: {probe_set_df['gene_id'][probe_set_df['included'] == True].nunique():,}")
print(f"Number of non-included gene IDs in table: {probe_set_df['gene_id'][probe_set_df['included'] == False].nunique():,}")

print(f"\nTotal number of probe IDs: {probe_set_df['probe_id'].nunique():,}")
print(f"Total number of `spliced` probes: {probe_set_df['probe_id'][probe_set_df['region'] == 'spliced'].nunique():,}")
print(f"Total number of `unspliced` probes: {probe_set_df['probe_id'][probe_set_df['region'] == 'unspliced'].nunique():,}")

Number of gene IDs in table: 19,017
Number of included gene IDs in table: 18,085
Number of non-included gene IDs in table: 932

Total number of probe IDs: 54,580
Total number of `spliced` probes: 7,605
Total number of `unspliced` probes: 46,975


In [45]:
# Number of probes per gene (all)
probes_per_gene = (
    probe_set_df
    .groupby("gene_id")["probe_id"]
    .nunique()
)
genes_by_probe_count = probes_per_gene.value_counts().sort_index()

for n_probes, n_genes in genes_by_probe_count.items():
    print(f"Genes with {n_probes} probes: {n_genes:,}")

Genes with 1 probes: 1,771
Genes with 2 probes: 961
Genes with 3 probes: 15,359
Genes with 4 probes: 330
Genes with 5 probes: 196
Genes with 6 probes: 351
Genes with 7 probes: 16
Genes with 8 probes: 8
Genes with 9 probes: 24
Genes with 12 probes: 1


In [46]:
# Subset data to included genes
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['square_002um'][:, sdata.tables['square_002um'].var['gene_ids'].isin(included)]

In [47]:
# Total number of spots in table with at least one UMI (all)
n_spots_not_zero = (row_sums(sdata.tables['square_002um'].X) > 0).sum()
print(f"Number of spots in table with at least one UMI (all): {n_spots_not_zero:,}")

Number of spots in table with at least one UMI (all): 9,419,791


In [48]:
# Total number of spots in table with at least one UMI (included genes)
n_spots_not_zero = (row_sums(adata.X) > 0).sum()
print(f"Number of spots in table with at least one UMI (included genes): {n_spots_not_zero:,}")

Number of spots in table with at least one UMI (included genes): 9,374,513


In [49]:
# Number of spots in tissue with at least one UMI (all)
in_tissue_mask = sdata.tables['square_002um'].obs['in_tissue_annotation'] == 1
umis_mask = row_sums(sdata.tables['square_002um'].X) > 0
n_spots_in_tissue_not_zero = (in_tissue_mask & umis_mask).sum()
print(f"Number of spots in tissue with at least one UMI (all): {n_spots_in_tissue_not_zero:,}")

Number of spots in tissue with at least one UMI (all): 9,100,058


In [50]:
# Number of spots in tissue with at least one UMI (included genes)
in_tissue_mask = adata.obs['in_tissue_annotation'] == 1
umis_mask = row_sums(adata.X) > 0
n_spots_in_tissue_not_zero = (in_tissue_mask & umis_mask).sum()
print(f"Number of spots in tissue with at least one UMI (included genes): {n_spots_in_tissue_not_zero:,}")

Number of spots in tissue with at least one UMI (included genes): 9,062,973


In [51]:
# Total number of genes targeted
print(f"Total number of genes targeted: {len(included):,}")

Total number of genes targeted: 18,085


In [52]:
# Total number of genes in table (all)
print(f"Total number of genes in table (all): {(col_sums(sdata.tables['square_002um'].X) > 0).sum():,}")

Total number of genes in table (all): 18,158


In [53]:
# Total number of genes in table (included genes)
print(f"Total number of genes in table (included genes): {(col_sums(adata.X) > 0).sum():,}")

Total number of genes in table (included genes): 17,386


In [54]:
# Number of genes in tissue (all)
print(f"Number of genes in tissue (all): {(col_sums(sdata.tables['square_002um'][sdata.tables['square_002um'].obs['in_tissue_annotation'] == 1].X) > 0).sum():,}")

Number of genes in tissue (all): 18,151


In [55]:
# Number of genes in tissue (included genes)
print(f"Number of genes in tissue (included genes): {(col_sums(adata[adata.obs['in_tissue_annotation'] == 1].X) > 0).sum():,}")

Number of genes in tissue (included genes): 17,380


In [56]:
# Total number of unique UMIs (all)
print(f"Total number of unique UMIs (all): {sdata.tables['square_002um'].X.sum():,.0f}")

Total number of unique UMIs (all): 76,173,472


In [57]:
# Total number of unique UMIs (included genes)
print(f"Total number of unique UMIs (included genes): {adata.X.sum():,.0f}")

Total number of unique UMIs (included genes): 73,230,408


In [58]:
# Number of unique UMIs in tissue (all)
print(f"Number of unique UMIs in tissue (all): {sdata.tables['square_002um'][sdata.tables['square_002um'].obs['in_tissue_annotation'] == 1].X.sum():,.0f}")

Number of unique UMIs in tissue (all): 75,656,936


In [59]:
# Number of unique UMIs in tissue (included genes)
print(f"Number of unique UMIs in tissue (included genes): {adata[adata.obs['in_tissue_annotation'] == 1].X.sum():,.0f}")

Number of unique UMIs in tissue (included genes): 72,732,920


In [60]:
import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for all indices
    total_sum = np.sum(count)
    total_count = len(count)

    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum_included = np.sum(included_counts)
    total_count_included = len(included_counts)

    print(f"Total number of UMIs (all): {total_sum:,}")
    print(f"Total number of unique UMIs (all): {total_count:,}")

    print(f"\nTotal number of UMIs (included genes): {total_sum_included:,}")
    print(f"Total number of unique UMIs (included genes): {total_count_included:,}")


Total number of UMIs (all): 90,049,808
Total number of unique UMIs (all): 76,173,467

Total number of UMIs (included genes): 86,568,300
Total number of unique UMIs (included genes): 73,230,409
